# 05. Transformer Archetypes: BERT, GPT, T5, ViT, Swin, Perceiver, and Jamba

- **Category:** 05_architectures / 04_transformers_and_heads
- **Lineage & Citations:** Jacob Devlin et al., 2018 (*BERT*); Alexey Dosovitskiy et al., 2020 (*ViT*); Ze Liu et al., 2021 (*Swin Transformer*); Andrew Jaegle et al., 2021 (*Perceiver*); Opher Lieber et al., 2024 (*Jamba*).
- **Purpose:** Surveys foundational transformer topological archetypes—from bidirectional encoders (BERT) and causal decoders (GPT) to asymmetric cross-attention bottlenecks (Perceiver) and hybrid SSM-Transformers (Jamba).
- **Topic Connections:** `🔗 Upstream Prerequisites:` [Sparse Sliding Window and Sink Attention](04_sparse_sliding_window_and_sink_attention.ipynb) $\to$ `🔗 Downstream Syntheses:` [Autoencoders VAE Beta VAE and VQ FSQ LFQ](../05_generative_models/01_autoencoders_vae_beta_vae_and_vq_fsq_lfq.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> - **BERT for Autoregressive Text Generation:** Bidirectional models have no causal training signal and produce degenerate text when unrolled autoregressively.
> - **Perceiver for Fine-Grained Segmentation:** Heavy latent bottleneck compression destroys pixel-precise spatial coordinates.

## Mathematical Foundations & Governing Equations

### 1. Perceiver Asymmetric Cross-Attention Bottleneck
Distills $M$ high-dimensional inputs into $N$ latent slots ($N \ll M$):
$$
Q = L W_Q \in \mathbb{R}^{B \times N \times D}, \quad K = X W_K, \ V = X W_V \in \mathbb{R}^{B \times M \times D}
$$
$$
A = \text{softmax}\left(\frac{Q K^\top}{\sqrt{D}}\right) \in \mathbb{R}^{B \times N \times M}, \quad L_{\text{out}} = (A V) W_O \in \mathbb{R}^{B \times N \times D}
$$

### 2. Causal Autoregressive Masking
$$
M_{i, j} = \begin{cases} 0 & \text{if } j \le i \\ -\infty & \text{if } j > i \end{cases}, \quad A_{\text{causal}} = \text{softmax}(S + M)
$$

In [ ]:
import math
import torch
import torch.nn.functional as F
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)
CKPT_REL = Path("weights/05_architectures/04_transformers_and_heads/05_transformer_archetypes_bert_gpt_t5_vit_swin_perceiver_and_jamba.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)

latents = weights["latents"]
inputs = weights["inputs"]
scale = float(weights["scale"][0])
print(f"Loaded {len(weights)} golden tensors on {env.device}, latents: {list(latents.shape)}, inputs: {list(inputs.shape)}")

In [ ]:
def perceiver_cross_attention(latents, inputs, w_q, w_k, w_v, w_o, scale):
    q = latents @ w_q
    k = inputs @ w_k
    v = inputs @ w_v
    scores = torch.bmm(q, k.transpose(1, 2)) * scale
    attn_weights = F.softmax(scores, dim=-1)
    return (torch.bmm(attn_weights, v)) @ w_o, attn_weights

def causal_decoder_mask(scores):
    T = scores.shape[-1]
    mask = torch.tril(torch.ones(T, T, device=scores.device, dtype=torch.bool))
    return torch.where(mask.unsqueeze(0), scores, torch.full_like(scores, -1e9))

In [ ]:
# Quantitative Mathematical Assertions & Golden Parity Verification
p_out, p_attn = perceiver_cross_attention(
    latents, inputs, weights["w_q"], weights["w_k"], weights["w_v"], weights["w_o"], scale
)
err_p = float(torch.max(torch.abs(p_out - weights["perceiver_out"])))
err_pa = float(torch.max(torch.abs(p_attn - weights["perceiver_attn"])))
assert err_p < 1e-6, f"Perceiver output error: {err_p:.2e}"
assert err_pa < 1e-6, f"Perceiver attention error: {err_pa:.2e}"

c_scores = causal_decoder_mask(weights["raw_scores"])
err_c = float(torch.max(torch.abs(c_scores - weights["causal_scores"])))
assert err_c < 1e-6, f"Causal scores error: {err_c:.2e}"
print(f"Parity Verified: Perceiver Out={err_p:.2e}, Attn={err_pa:.2e}, Causal Scores={err_c:.2e}")

In [ ]:
import matplotlib.pyplot as plt
from core.benchmark import benchmark_fn, format_benchmark_table

b_perceiver = benchmark_fn(lambda: perceiver_cross_attention(
    latents, inputs, weights["w_q"], weights["w_k"], weights["w_v"], weights["w_o"], scale
), iters=50, name="Perceiver Bottleneck")
b_causal = benchmark_fn(lambda: causal_decoder_mask(weights["raw_scores"]), iters=50, name="Causal Masking")
print(format_benchmark_table([b_perceiver, b_causal]))

# Visualization: Perceiver Cross-Attention Map & Causal Triangle
fig, ax = plt.subplots(1, 2, figsize=(8, 2.5), dpi=100)
ax[0].imshow(p_attn[0].detach().numpy(), cmap="viridis", aspect="auto")
ax[0].set_title("Perceiver Cross-Attn [N_latents, M_inputs]")
ax[1].imshow(c_scores[0].detach().numpy(), cmap="plasma", aspect="auto")
ax[1].set_title("Causal Decoder Mask Scores [T, T]")
plt.tight_layout()
plt.show()

In [ ]:
# Pathological Misuse: Reversed Cross-Attention Directionality (M x N instead of N x M)
# If inputs query latents, the output shape expands to M instead of compressing to N!
scores_reversed = torch.bmm(inputs @ weights["w_k"], (latents @ weights["w_q"]).transpose(1, 2))
print(f"Correct Bottleneck Attention Shape: {list(p_attn.shape)}")
print(f"Reversed Attention Shape: {list(scores_reversed.shape)}")
assert scores_reversed.shape[1] == inputs.shape[1], "Reversed query fails to compress input dimension!"
print("Demonstrated requirement of latent queries for asymmetric Perceiver bottlenecking.")

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Directionality Defines Compression:** Perceiver queries originating from latents decouple compute cost from raw input size.
2. **Causal Masking Enforces Autoregressive Generation:** Lower-triangular masking prevents future token leakage in decoders.
3. **Hybrid Architectures Unify Paradigms:** Jamba-style models combine Transformer memory with SSM speed.

---
### Cross-Domain Generalisation Matrix
- **Heterogeneous Multimodal Sensing:** Perceiver digests lidar, audio waveforms, and pixels simultaneously.
- **Hybrid Long-Context Serving:** Jamba balances KV cache reductions with full associative recall.
- **Scalable Vision Backbones:** Swin and ViT replace inductive CNN kernels with global patch attention.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [05_transformer_archetypes_bert_gpt_t5_vit_swin_perceiver_and_jamba.md](../../../markdown/05_architectures/04_transformers_and_heads/05_transformer_archetypes_bert_gpt_t5_vit_swin_perceiver_and_jamba.md)
- **PyTorch 2.x Module:** [05_transformer_archetypes_bert_gpt_t5_vit_swin_perceiver_and_jamba.py](../../../pytorch2/05_architectures/04_transformers_and_heads/05_transformer_archetypes_bert_gpt_t5_vit_swin_perceiver_and_jamba.py)
- **Functional JAX Module:** [05_transformer_archetypes_bert_gpt_t5_vit_swin_perceiver_and_jamba.py](../../../jax/05_architectures/04_transformers_and_heads/05_transformer_archetypes_bert_gpt_t5_vit_swin_perceiver_and_jamba.py)
- **Candle Rust Module:** [05_transformer_archetypes_bert_gpt_t5_vit_swin_perceiver_and_jamba.rs](../../../candle/05_architectures/04_transformers_and_heads/05_transformer_archetypes_bert_gpt_t5_vit_swin_perceiver_and_jamba.rs)
- **Hardware Kernel:** [05_transformer_archetypes_bert_gpt_t5_vit_swin_perceiver_and_jamba.rs](../../../kernels/05_architectures/04_transformers_and_heads/05_transformer_archetypes_bert_gpt_t5_vit_swin_perceiver_and_jamba.rs)
- **Universal Golden SafeTensors:** [05_transformer_archetypes_bert_gpt_t5_vit_swin_perceiver_and_jamba.safetensors](../../../weights/05_architectures/04_transformers_and_heads/05_transformer_archetypes_bert_gpt_t5_vit_swin_perceiver_and_jamba.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous Topic: 04. Sparse Sliding Window and Sink Attention](04_sparse_sliding_window_and_sink_attention.ipynb) | [Category Index](../../../README.md#05_architectures) | [Next Topic: 01. Autoencoders VAE Beta VAE and VQ FSQ LFQ](../05_generative_models/01_autoencoders_vae_beta_vae_and_vq_fsq_lfq.ipynb) &rarr;